<a href="https://colab.research.google.com/github/rund-omari/Data.Eng3/blob/main/lectures/Lec14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
data = pd.read_csv('/content/SDK_Hotel_Bookings_TEACHING_SAMPLE.csv')

In [ ]:
data.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_week_nights,stays_in_weekend_nights,adults,children,babies,adr,market_segment,deposit_type,reservation_status,reservation_status_date
0,Resort Hotel,1,0,2015,January,1,1,2,3,2.0,0,192.04,Corporate,Refundable,Canceled,2015-01-01
1,City Hotel,0,1,2015,February,8,5,3,3,1.0,0,55.39,Online TA,No Deposit,Check-Out,2015-02-08
2,City Hotel,0,4,2015,March,15,3,0,2,0.0,0,160.31,Direct,No Deposit,Check-Out,2015-03-15
3,Resort Hotel,0,10,2015,April,22,3,2,2,NaN,1,153.85,Groups,No Deposit,Check-Out,2015-04-22
4,City Hotel,0,18,2015,May,4,5,2,2,1.0,0,190.85,Groups,No Deposit,Check-Out,2015-05-04


In [ ]:
month_number = pd.to_datetime(data["arrival_date_month"], format="%B", errors="coerce").dt.month
month_number

,arrival_date_month
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10


In [ ]:
data['arrival_date'] = pd.to_datetime({
    "year": data["arrival_date_year"],
    "month": month_number,
    "day": data["arrival_date_day_of_month"]
}, errors="coerce")

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   hotel                      47 non-null     object        
 1   is_canceled                48 non-null     int64         
 2   lead_time                  48 non-null     int64         
 3   arrival_date_year          48 non-null     int64         
 4   arrival_date_month         48 non-null     object        
 5   arrival_date_day_of_month  48 non-null     int64         
 6   stays_in_week_nights       48 non-null     int64         
 7   stays_in_weekend_nights    48 non-null     int64         
 8   adults                     48 non-null     int64         
 9   children                   46 non-null     float64       
 10  babies                     48 non-null     int64         
 11  adr                        48 non-null     float64       
 12  market_seg

In [ ]:
data.lead_time.describe()

,lead_time
count,48.00000
mean,66.31250
std,74.38547
min,-5.00000
25%,18.75000
50%,43.50000
75%,90.25000
max,365.00000


In [ ]:
data.lead_time

In [ ]:
bins = [-5, 100, 150, 200,250,300,350,400]
labels = ['b1' ,'b2','b3','b4','b5','b6','b7']

data["lead_time_group"] = pd.cut(
    data["lead_time"], bins=bins, labels=labels,
    right=True, include_lowest=True)

In [ ]:
data[["lead_time", "lead_time_group"]].sample(12)

,lead_time,lead_time_group
0,0,b1
1,1,b1
2,4,b1
3,10,b1
4,18,b1
5,28,b1
6,29,b1
7,30,b1
8,30,b1
9,31,b1


In [ ]:
data["lead_time_group"].value_counts(dropna=False)

,count
lead_time_group,
b1,40
b2,4
b3,1
b4,1
b5,1
b7,1
b6,0


In [ ]:
valid_leads = data.loc[data["lead_time"].ge(0), "lead_time"].dropna()
valid_leads
quantile_groups, edges = pd.qcut(valid_leads, q=4, duplicates="drop", retbins=True)
print("Actual quantile edges:", edges)
print(quantile_groups.value_counts(sort=False))

,lead_time
0,0
1,1
2,4
3,10
4,18
5,28
6,29
7,30
8,30
9,31


In [ ]:
data.deposit_type.unique()

array(['Refundable', 'No Deposit'], dtype=object)

In [ ]:
mean_adr_by_hotel = data.groupby(["hotel",'deposit_type'])["adr"].mean()
mean_adr_by_hotel

hotel         deposit_type
City Hotel    No Deposit      126.916500
              Refundable      153.411667
Resort Hotel  No Deposit      130.308000
              Refundable      594.946000
Name: adr, dtype: float64

In [ ]:
#orders["city_clean"] = orders["city"].astype("string").str.strip().str.title().fillna("Unknown")

In [ ]:
mean_adr_by_hotel = data.groupby(["hotel",'deposit_type']).agg({"adr": ["mean", "median"],
                                                                'children' : 'min',
                                                                'adults': 'max'})
mean_adr_by_hotel

adr          children adults
                                 mean   median      min    max
hotel        deposit_type                                     
City Hotel   No Deposit    126.916500  145.795      0.0      3
             Refundable    153.411667  150.665      0.0      3
Resort Hotel No Deposit    130.308000  132.390      0.0      3
             Refundable    594.946000  185.890      0.0      3